# Categorical Cross‑Entropy (CCE)

In [1]:
import tensorflow as tf
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import pickle

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

2026-06-08 05:05:48.759773: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-08 05:05:48.759892: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-08 05:05:48.763965: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-08 05:05:48.783249: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 Laptop GPU, compute capability 8.6


2026-06-08 05:05:52.020896: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 05:05:52.045793: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 05:05:52.045848: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 05:05:52.046229: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


TensorFlow version: 2.15.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
organized_base = pathlib.Path("/tf/work/data/organized_dataset")
train_dir = organized_base / "Train"
val_dir = organized_base / "Validation"
test_dir = organized_base / "Test"

IMG_SIZE = (128, 128)
BATCH_SIZE = 8

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Train: {train_gen.samples}, Val: {val_gen.samples}, Test: {test_gen.samples}")

Found 49376 images belonging to 15 classes.
Found 10584 images belonging to 15 classes.
Found 10589 images belonging to 15 classes.
Train: 49376, Val: 10584, Test: 10589


In [3]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(train_gen.num_classes, activation='softmax', dtype='float32')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

2026-06-08 05:06:01.543880: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 05:06:01.543972: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 05:06:01.544006: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 05:06:01.785099: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-08 05:06:01.785273: I external/local_xla/xla/stream_executor

58889256/58889256 [==============================] - 10s 0us/step
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 128, 128, 3)]     0         
                                                                 
 vgg16 (Functional)          (None, 4, 4, 512)         14714688  
                                                                 
 global_average_pooling2d (  (None, 512)               0         
 GlobalAveragePooling2D)                                         
                                                                 
 dropout (Dropout)           (None, 512)               0         
                                                                 
 dense (Dense)               (None, 15)                7695      
                                                                 
Total params: 14722383 (56.16 MB)
Trainable params: 7695 (30.

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1)
]

history = model.fit(
    train_gen,
    steps_per_epoch=train_gen.samples // BATCH_SIZE,
    epochs=20,
    validation_data=val_gen,
    validation_steps=val_gen.samples // BATCH_SIZE,
    callbacks=callbacks,
    workers=4,
    use_multiprocessing=True
)

with open('history_cce.pkl', 'wb') as f:
    pickle.dump(history.history, f)

Epoch 1/20


2026-06-08 05:06:24.327834: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8906
2026-06-08 05:06:26.408465: I external/local_xla/xla/service/service.cc:168] XLA service 0x73424c003930 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-06-08 05:06:26.408542: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-06-08 05:06:26.437768: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780895186.608790     648 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


6172/6172 [==============================] - 217s 35ms/step - loss: 1.9800 - accuracy: 0.3896 - val_loss: 1.5578 - val_accuracy: 0.4956 - lr: 1.0000e-04
Epoch 2/20
6172/6172 [==============================] - 209s 34ms/step - loss: 1.5067 - accuracy: 0.5315 - val_loss: 1.2397 - val_accuracy: 0.6169 - lr: 1.0000e-04
Epoch 3/20
6172/6172 [==============================] - 205s 33ms/step - loss: 1.2897 - accuracy: 0.6024 - val_loss: 1.0661 - val_accuracy: 0.6727 - lr: 1.0000e-04
Epoch 4/20
6172/6172 [==============================] - 216s 35ms/step - loss: 1.1570 - accuracy: 0.6446 - val_loss: 0.9450 - val_accuracy: 0.7184 - lr: 1.0000e-04
Epoch 5/20
6172/6172 [==============================] - 210s 34ms/step - loss: 1.0707 - accuracy: 0.6704 - val_loss: 0.8612 - val_accuracy: 0.7447 - lr: 1.0000e-04
Epoch 6/20
6172/6172 [==============================] - 214s 35ms/step - loss: 1.0042 - accuracy: 0.6891 - val_loss: 0.8092 - val_accuracy: 0.7558 - lr: 1.0000e-04
Epoch 7/20
6172/6172 [=====

In [ ]:
test_loss, test_acc = model.evaluate(test_gen, workers=4, use_multiprocessing=True)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
class_names = list(train_gen.class_indices.keys())
class_counts = [np.sum(train_gen.classes == i) for i in range(len(class_names))]

plt.figure(figsize=(12,5))
plt.bar(class_names, class_counts, color='skyblue')
plt.xticks(rotation=45, ha='right')
plt.title('Training Set Class Distribution – Severe Imbalance')
plt.ylabel('Number of images')
plt.xlabel('Fruit class')
plt.grid(axis='y', alpha=0.3)
plt.show()

min_count = min(class_counts)
max_count = max(class_counts)
print(f"Majority class (largest): {class_names[np.argmax(class_counts)]} with {max_count} images")
print(f"Minority class (smallest): {class_names[np.argmin(class_counts)]} with {min_count} images")
print(f"Ratio majority/minority: {max_count/min_count:.1f}:1")

In [ ]:
y_pred_probs = model.predict(test_gen)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

print("\n" + "="*60)
print("CLASSIFICATION REPORT – CCE BASELINE")
print("="*60)
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(14,12))
sns.heatmap(cm_norm, annot=False, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Normalised Confusion Matrix – CCE Baseline')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

In [ ]:
recalls = recall_score(y_true, y_pred, average=None)
minority_threshold = 2000
minority_indices = [i for i, cnt in enumerate(class_counts) if cnt < minority_threshold]
minority_recalls = [recalls[i] for i in minority_indices]

print(f"\nMinority classes (<{minority_threshold} images):")
for idx in minority_indices:
    print(f"  {class_names[idx]}: recall = {recalls[idx]:.4f}")
print(f"\nAverage recall on minority classes: {np.mean(minority_recalls):.3f}")
print(f"Average recall on majority classes: {np.mean([recalls[i] for i, cnt in enumerate(class_counts) if cnt >= minority_threshold]):.3f}")
print("\n🔴 PROBLEM 1 – CLASS IMBALANCE: Minority classes perform much worse.")

In [ ]:
misclassified_indices = np.where(y_pred != y_true)[0]
confusion_pairs = [(y_true[i], y_pred[i]) for i in misclassified_indices]
pair_counts = Counter(confusion_pairs)

print("\n" + "="*60)
print("TOP 5 MOST CONFUSED FRUIT PAIRS (true → predicted)")
print("="*60)
for (true, pred), count in pair_counts.most_common(5):
    print(f"{class_names[true]:15s} → {class_names[pred]:15s} : {count} errors")
print("\n🔴 PROBLEM 2 – HARD EXAMPLES: Similar fruits are often confused.")

In [ ]:
metrics = {
    'classification_report': report,
    'confusion_matrix': cm,
    'class_names': class_names,
    'class_counts': class_counts,
    'minority_recalls': minority_recalls,
    'confusion_pairs': pair_counts.most_common(10)
}
with open('cce_metrics.pkl', 'wb') as f:
    pickle.dump(metrics, f)
print("Metrics saved to 'cce_metrics.pkl'")